In [ ]:
from utils import calculate_nse, create_lag, create_lag_adv, create_lag_adv_corrected

In [ ]:
from shap import initjs


initjs()

In [ ]:
#tips
#for single column dataframes always use double brackets, single brackets will make it a series, or float type
# for adding two columns to a df either add them one by one or use two brackets

In [ ]:
import pandas as pd
df = pd.read_csv('final_data_physics_baseline/phy_final_corrected_data.csv')
pd.to_datetime(df['date'])
df = df.set_index('date')
df.drop(columns= 'Unnamed: 0', inplace= True)
df.drop(columns= 'Unnamed: 0.1', inplace= True)

# we convert 0-12 to radians circle for a seasonal mapping (circular time encoding)
# ensure index is treated as datetime (works even if the Index is not recognized as DatetimeIndex)
import numpy as np
month = pd.DatetimeIndex(df.index).month

df['year_sin'] = np.sin(2*np.pi*month/12)
df['year_cos'] = np.cos(2*np.pi*month/12)


df1 = df.copy()
df.head()


In [ ]:
# #WITHOUT SEASON AWARENESS
# #train baseline prediction regression model only fitted on train data
# from sklearn.preprocessing import StandardScaler
# from sklearn.linear_model import LinearRegression
# # lag_scaler = StandardScaler()


# x_lagged = create_lag(df1, lag_precip=1, lag_dd=3, lag_sca=3, lag_et=1)

# x_train = x_lagged[:'2017-12-31']
# x_val = x_lagged['2018-01-01':'2021-12-31']
# x_test = x_lagged['2022-01-01':]

# y_train = x_lagged[:'2017-12-31']
# y_val = x_lagged['2018-01-01':'2021-12-31']
# y_test = x_lagged['2022-01-01':]

# X_train_lagged = x_train[[ 'sca_lagged','dd_lagged','melt_proxy', 'et_loss_lagged', 'precipitation_lagged']]
# Y_train_lagged = y_train[['runoff']]
# Y_val_lagged = y_val[['runoff']]
# X_val_lagged = x_val[[ 'sca_lagged','dd_lagged','melt_proxy', 'et_loss_lagged', 'precipitation_lagged']]
# X_test_lagged = x_test[['sca_lagged','dd_lagged','melt_proxy', 'et_loss_lagged', 'precipitation_lagged']]
# Y_test_lagged = x_test[['runoff']]

# X_train_val = pd.concat([X_train_lagged, X_val_lagged])
# Y_train_val = pd.concat([Y_train_lagged, Y_val_lagged])
# x_lagged.head()
# # lag_scaler.fit(X_train_lagged)
 
# # x_train_scaled = lag_scaler.transform(X_train_lagged)
# # x_val_scaled = lag_scaler.transform(X_val_lagged)
# # x_test_scaled = lag_scaler.transform(X_test_lagged)

# model = LinearRegression()
# model.fit(X_train_lagged, Y_train_lagged)

# y_test_pred = model.predict(X_test_lagged)
# baseline_nse = calculate_nse(Y_test_lagged, y_test_pred)
# print(baseline_nse)
# # alpha, beta, gamma, delta=amodel.coef_
# print(model.coef_)

In [ ]:
#train baseline prediction regression model only fitted on train data
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
# lag_scaler = StandardScaler()


x_lagged = create_lag(df1, lag_precip=1, lag_dd=3, lag_sca=3, lag_et=1)

x_train = x_lagged[:'2017-12-31']
x_val = x_lagged['2018-01-01':'2021-12-31']
x_test = x_lagged['2022-01-01':]

y_train = x_lagged[:'2017-12-31']
y_val = x_lagged['2018-01-01':'2021-12-31']
y_test = x_lagged['2022-01-01':]

X_train_lagged = x_train[['sca_lagged','dd_lagged','year_sin', 'year_cos', 'melt_proxy', 'et_loss_lagged', 'precipitation_lagged']]
Y_train_lagged = y_train[['runoff']]
Y_val_lagged = y_val[['runoff']]
X_val_lagged = x_val[['sca_lagged','dd_lagged','year_sin', 'year_cos', 'melt_proxy', 'et_loss_lagged', 'precipitation_lagged']]
X_test_lagged = x_test[['sca_lagged','dd_lagged','year_sin', 'year_cos', 'melt_proxy', 'et_loss_lagged', 'precipitation_lagged']]
Y_test_lagged = x_test[['runoff']]

X_train_val = pd.concat([X_train_lagged, X_val_lagged])
Y_train_val = pd.concat([Y_train_lagged, Y_val_lagged])
x_lagged.head()
# lag_scaler.fit(X_train_lagged)
 
# x_train_scaled = lag_scaler.transform(X_train_lagged)
# x_val_scaled = lag_scaler.transform(X_val_lagged)
# x_test_scaled = lag_scaler.transform(X_test_lagged)

seasonaware_model = LinearRegression()
seasonaware_model.fit(X_train_lagged, Y_train_lagged)

y_test_pred = seasonaware_model.predict(X_test_lagged)
baseline_nse = calculate_nse(Y_test_lagged, y_test_pred)
print(baseline_nse)
# alpha, beta, gamma, delta=amodel.coef_
print(seasonaware_model.coef_)

In [ ]:
train_preds = seasonaware_model.predict(X_train_lagged)
val_preds = seasonaware_model.predict(X_val_lagged)
test_preds = seasonaware_model.predict(X_test_lagged)

#recombination
baseline_q_no_leak = pd.concat([
    pd.DataFrame(train_preds, index=X_train_lagged.index, columns=['baseline_q']),
    pd.DataFrame(val_preds, index=X_val_lagged.index, columns=['baseline_q']),
    pd.DataFrame(test_preds, index=X_test_lagged.index, columns=['baseline_q'])
])
Y_full = pd.concat([Y_train_val, Y_test_lagged])

df_full = pd.concat([X_train_lagged, X_val_lagged, X_test_lagged])
df_full['baseline_q'] = baseline_q_no_leak
df_full['residuals'] = Y_full['runoff'] - df_full['baseline_q']

df_adv = create_lag_adv_corrected(df1) 
df_adv = df_adv.join(df_full[['baseline_q', 'residuals']]).dropna()

In [ ]:
# from xgboost import XGBRegressor
# import pandas as pd

# xgb = XGBRegressor(
#     max_depth=5,
#     learning_rate=0.01,
#     n_estimators=1500,
#     random_state=42, # Add for reproducibility
# )

# train_ml = df_adv[:'2017-12-31']
# val_ml = df_adv['2018-01-01':'2021-12-31']
# test_ml = df_adv['2022-01-01':]

# X_train_ml = train_ml.drop(columns=['baseline_q', 'residuals'])
# y_train_ml = train_ml['residuals']

# X_val_ml = val_ml.drop(columns=['baseline_q', 'residuals'])
# y_val_ml = val_ml['residuals']

# # 4. Fit the model ONLY on the training data
# xgb.fit(X_train_ml, y_train_ml)

# # 5. Predict on the validation set
# residual_pred_val = xgb.predict(X_val_ml)
# q_hybrid_val = residual_pred_val + val_ml['baseline_q']

# # 6. Evaluate against the correct validation ground truth
# y_val_true = Y_full['runoff'].loc[val_ml.index]
# hybrid_nse_val = calculate_nse(y_val_true, q_hybrid_val)

# print(f"Hybrid NSE (Validation): {hybrid_nse_val:.4f}")


In [ ]:
# # --- Final Test Set Evaluation --- train model on train + val data

# # Prepare combined training/validation data
# X_train_val_ml = pd.concat([X_train_ml, X_val_ml])
# y_train_val_ml = pd.concat([y_train_ml, y_val_ml])

# # Prepare test data
# X_test_ml = test_ml.drop(columns=['baseline_q', 'residuals'])

# # Re-fit the model on all available training data
# xgb.fit(X_train_val_ml, y_train_val_ml)

# # Predict on the test set
# residual_pred_test = xgb.predict(X_test_ml)
# q_hybrid_test = residual_pred_test + test_ml['baseline_q']

# # Evaluate against the test ground truth
# y_test_true = Y_full['runoff'].loc[test_ml.index]
# hybrid_nse_test = calculate_nse(y_test_true, q_hybrid_test)

# print(f"Final Hybrid NSE (Test): {hybrid_nse_test:.4f}")

In [ ]:

df_adv = create_lag_adv_corrected(df1) 
df_adv = df_adv.join(df_full[['baseline_q', 'residuals']]).dropna()
df_adv.head()
# df_adv['year_sin'] = x_lagged['year_sin']
# df_adv['year_cos'] = x_lagged['year_cos']
col_names = df_adv.columns.tolist()
col_names

In [ ]:

# df_adv = create_lag_adv_corrected(df1) 
# df_adv = df_adv.join(df_full[['baseline_q', 'residuals']]).dropna()
# from xgboost import XGBRegressor
# import pandas as pd

# # --- FIX: Explicitly define the feature set to exclude current-day data ---
# # These are the features created by create_lag_adv_corrected
# lagged_feature_columns = [
#     'sca_lagged_1', 'sca_lagged_2',
#     'dd_lagged_1', 'dd_lagged_2',
#     'et_loss_lagged_1',
#     'precipitation_lagged_1',
#     'melt_proxy_1_1', 'melt_proxy_1_2',
#     'melt_proxy_2_1', 'melt_proxy_2_2'
# ]

# # These are the columns we want to predict or use in the final calculation
# target_and_baseline_cols = ['baseline_q', 'residuals']

# # The final dataframe for the ML model should ONLY contain these columns
# df_ml_final = df_adv[lagged_feature_columns+ target_and_baseline_cols]


# # --- Now proceed with the corrected dataframe ---
# xgb1 = XGBRegressor(
#     max_depth=5,
#     learning_rate=0.03,
#     n_estimators=1500,
#     random_state=42
# )

# # 1. Split the clean ML dataframe
# train_ml = df_ml_final[:'2017-12-31']
# val_ml = df_ml_final['2018-01-01':'2021-12-31']
# test_ml = df_ml_final['2022-01-01':]

# # 2. Prepare features and labels for validation
# X_train_ml = train_ml.drop(columns=target_and_baseline_cols)
# y_train_ml = train_ml['residuals']

# X_val_ml = val_ml.drop(columns=target_and_baseline_cols)

# # 3. Fit the model ONLY on the training data
# xgb1.fit(X_train_ml, y_train_ml)

# # 4. Predict on the validation set
# residual_pred_val = xgb1.predict(X_val_ml)
# q_hybrid_val = residual_pred_val + val_ml['baseline_q']

# # 5. Evaluate against the correct validation ground truth
# y_val_true = Y_full['runoff'].loc[val_ml.index]
# hybrid_nse_val = calculate_nse(y_val_true, q_hybrid_val)

# print(f"Corrected Hybrid NSE (Validation): {hybrid_nse_val:.4f}")

# # --- Final Test Set Evaluation ---
# # Prepare combined training/validation data
# X_train_val_ml = pd.concat([X_train_ml, X_val_ml])
# y_train_val_ml = pd.concat([y_train_ml, val_ml['residuals']])

# # Prepare test data
# X_test_ml = test_ml.drop(columns=target_and_baseline_cols)

# # Re-fit the model on all available training data
# xgb1.fit(X_train_val_ml, y_train_val_ml)

# # Predict on the test set
# residual_pred_test = xgb1.predict(X_test_ml)
# q_hybrid_test = residual_pred_test + test_ml['baseline_q']

# # Evaluate against the test ground truth
# y_test_true = Y_full['runoff'].loc[test_ml.index]
# hybrid_nse_test = calculate_nse(y_test_true, q_hybrid_test)

# print(f"Final Hybrid NSE (Test): {hybrid_nse_test:.4f}")

In [ ]:
df_features = df_adv.drop(columns=['residuals', 'baseline_q'])
df_features.head()

In [ ]:

# from xgboost import XGBRegressor
# import pandas as pd

# # --- FIX: Explicitly define the feature set to exclude current-day data ---
# # These are the features created by create_lag_adv_corrected
# lagged_feature_columns =[
#  'et_loss_lagged_1',
#  'precipitation_lagged_1',
#  'melt_proxy_1_1',
#  'melt_proxy_2_1',
#  'melt_proxy_2_2',
#  'melt_proxy_3_2',
#  'melt_proxy_3_3',
#  'melt_proxy_4_3',
#  'melt_proxy_4_4',
#  'melt_proxy_5_4',
#  'melt_proxy_5_5',
#  'melt_proxy_6_5',
#  'melt_proxy_6_6',
#  'melt_proxy_7_6',
#  'melt_proxy_7_7',
#  'baseline_q',
#  'residuals',
# ]

# # These are the columns we want to predict or use in the final calculation
# target_and_baseline_cols = ['baseline_q', 'residuals']

# # The final dataframe for the ML model should ONLY contain these columns
# df_ml_final = df_adv[lagged_feature_columns]


# # --- Now proceed with the corrected dataframe ---
# xgb1 = XGBRegressor(
#     max_depth=5,
#     learning_rate=0.01,
#     n_estimators=1500,
#     random_state=42
# )

# # 1. Split the clean ML dataframe
# train_ml = df_ml_final[:'2017-12-31']
# val_ml = df_ml_final['2018-01-01':'2021-12-31']
# test_ml = df_ml_final['2022-01-01':]

# # 2. Prepare features and labels for validation
# X_train_ml = train_ml.drop(columns=target_and_baseline_cols)
# y_train_ml = train_ml['residuals']

# X_val_ml = val_ml.drop(columns=target_and_baseline_cols)

# # 3. Fit the model ONLY on the training data
# xgb1.fit(X_train_ml, y_train_ml)

# # 4. Predict on the validation set
# residual_pred_valf = xgb1.predict(X_val_ml)
# q_hybrid_valf = residual_pred_valf + val_ml['baseline_q']

# # 5. Evaluate against the correct validation ground truth
# y_val_true = Y_full['runoff'].loc[val_ml.index]
# hybrid_nse_val = calculate_nse(y_val_true, q_hybrid_valf)

# print(f"Corrected Hybrid NSE (Validation): {hybrid_nse_val:.4f}")

# # --- Final Test Set Evaluation ---
# # Prepare combined training/validation data
# X_train_val_ml = pd.concat([X_train_ml, X_val_ml])
# y_train_val_ml = pd.concat([y_train_ml, val_ml['residuals']])

# # Prepare test data
# X_test_ml = test_ml.drop(columns=target_and_baseline_cols)

# # Re-fit the model on all available training data
# xgb1.fit(X_train_val_ml, y_train_val_ml)

# # Predict on the test set
# residual_pred_testf = xgb1.predict(X_test_ml)
# q_hybrid_testf = residual_pred_testf + test_ml['baseline_q']

# # Evaluate against the test ground truth
# y_test_true = Y_full['runoff'].loc[test_ml.index]
# hybrid_nse_test = calculate_nse(y_test_true, q_hybrid_testf)

# print(f"Final Hybrid NSE (Test): {hybrid_nse_test:.4f}")

In [ ]:

from xgboost import XGBRegressor
import pandas as pd

# --- FIX: Explicitly define the feature set to exclude current-day data ---
# These are the features created by create_lag_adv_corrected
lagged_feature_columns =[
 'precipitation_lagged',
 'sca_lagged_1',
 'dd_lagged_1',
 'sca_lagged_2',
 'dd_lagged_2',
 'sca_lagged_3',
 'dd_lagged_3',
 'sca_lagged_4',
 'dd_lagged_4',
 'sca_lagged_5',
 'dd_lagged_5',
 'sca_lagged_6',
 'dd_lagged_6',
 'sca_lagged_7',
 'dd_lagged_7',
 'et_loss_lagged_1',
 'precipitation_lagged_1',
 'melt_proxy_1_1',
 'melt_proxy_2_1',
 'melt_proxy_2_2',
 'melt_proxy_3_2',
 'melt_proxy_3_3',
 'melt_proxy_4_3',
 'melt_proxy_4_4',
 'melt_proxy_5_4',
 'melt_proxy_5_5',
 'melt_proxy_6_5',
 'melt_proxy_6_6',
 'melt_proxy_7_6',
 'melt_proxy_7_7',
 'baseline_q',
 'residuals',
]

# These are the columns we want to predict or use in the final calculation
target_and_baseline_cols = ['baseline_q', 'residuals']

# The final dataframe for the ML model should ONLY contain these columns
df_ml_final = df_adv[lagged_feature_columns]


# --- Now proceed with the corrected dataframe ---
xgb2 = XGBRegressor(
    max_depth=4,
    learning_rate=0.03,
    n_estimators=1500,
    random_state=42
)

# 1. Split the clean ML dataframe
train_ml = df_ml_final[:'2017-12-31']
val_ml = df_ml_final['2018-01-01':'2021-12-31']
test_ml = df_ml_final['2022-01-01':]

# 2. Prepare features and labels for validation
X_train_ml = train_ml.drop(columns=target_and_baseline_cols)
y_train_ml = train_ml['residuals']

X_val_ml = val_ml.drop(columns=target_and_baseline_cols)

# 3. Fit the model ONLY on the training data
xgb2.fit(X_train_ml, y_train_ml)

# 4. Predict on the validation set
residual_pred_valf = xgb2.predict(X_val_ml)
q_hybrid_valf = residual_pred_valf + val_ml['baseline_q']

# 5. Evaluate against the correct validation ground truth
y_val_true = Y_full['runoff'].loc[val_ml.index]
hybrid_nse_val = calculate_nse(y_val_true, q_hybrid_valf)

print(f"Corrected Hybrid NSE (Validation): {hybrid_nse_val:.4f}")

# --- Final Test Set Evaluation ---
# Prepare combined training/validation data
X_train_val_ml = pd.concat([X_train_ml, X_val_ml])
y_train_val_ml = pd.concat([y_train_ml, val_ml['residuals']])

# Prepare test data
X_test_ml = test_ml.drop(columns=target_and_baseline_cols)

# Re-fit the model on all available training data
xgb2.fit(X_train_val_ml, y_train_val_ml)

# Predict on the test set
residual_pred_testf = xgb2.predict(X_test_ml)
q_hybrid_testf = residual_pred_testf + test_ml['baseline_q']

# Evaluate against the test ground truth
y_test_true = Y_full['runoff'].loc[test_ml.index]
hybrid_nse_test = calculate_nse(y_test_true, q_hybrid_testf)

print(f"Final Hybrid NSE (Test): {hybrid_nse_test:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.stattools import durbin_watson

fig, axes = plt.subplots(1,2, figsize= (18, 4))
# plot_acf( residual_pred_valf, lags = 12, ax = axes[0], title = 'val_acf' )
plot_acf( test_ml['residuals'], lags = 12, ax = axes[0], title = 'baseline_acf' )
plot_acf(residual_pred_testf, lags=12, ax=axes[1], title='test_acf')

plt.savefig('acf_residuals.png', dpi=150)
plt.show()

# 2. Durbin-Watson statistic (quick scalar check)
dw_baseline = durbin_watson(test_ml['residuals'])
dw_hybrid   = durbin_watson(residual_pred_testf)

print(f"Durbin-Watson — Baseline: {dw_baseline:.3f} | Hybrid: {dw_hybrid:.3f}")
# ~2.0 = no autocorrelation, <1.5 = positive autocorrelation (bad), >2.5 = negative


In [ ]:
# print(X_val_ml.iloc[0])
# print(val_ml['residuals'].iloc[0])
val_ml.count()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.stattools import durbin_watson

fig, axes = plt.subplots(1,3, figsize= (18, 4))
plot_acf( residual_pred_valf, lags = 12, ax = axes[0], title = 'val_acf' )
plot_acf( test_ml['residuals'], lags = 12, ax = axes[1], title = 'baseline_acf' )
plot_acf(residual_pred_testf, lags=12, ax=axes[2], title='test_acf')

plt.savefig('acf_residuals.png', dpi=150)
plt.show()

# 2. Durbin-Watson statistic (quick scalar check)
dw_baseline = durbin_watson(test_ml['residuals'])
dw_hybrid   = durbin_watson(residual_pred_testf)

print(f"Durbin-Watson — Baseline: {dw_baseline:.3f} | Hybrid: {dw_hybrid:.3f}")
# ~2.0 = no autocorrelation, <1.5 = positive autocorrelation (bad), >2.5 = negative


In [ ]:
# xgb.importance(
#   model = xgb1,
#   feature_names = getinfo(xgb1,
#     'sca_lagged_1',
#  'sca_lagged_2',
#  'sca_lagged_3',
#  'sca_lagged_4',
#  'sca_lagged_5',
#  'sca_lagged_6',
#  'sca_lagged_7',
#  'dd_lagged_1',
#  'dd_lagged_2',
#  'dd_lagged_3',
#  'dd_lagged_4',
#  'dd_lagged_5',
#  'dd_lagged_6',
#  'dd_lagged_7',
#  'et_loss_lagged_1',
#  'precipitation_lagged_1',
#  'melt_proxy_1_1',
#  'melt_proxy_1_2',
#  'melt_proxy_2_1',
#  'melt_proxy_2_2',
#  'melt_proxy_1_3',
#  'melt_proxy_1_4',
#  'melt_proxy_1_5',
#  'melt_proxy_1_6',
#  'melt_proxy_1_7',
#  'melt_proxy_2_3',
#  'melt_proxy_2_4',
#  'melt_proxy_2_5',
#  'melt_proxy_2_6',
#  'melt_proxy_2_7',
#  'melt_proxy_3_1',
#  'melt_proxy_3_2',
#  'melt_proxy_3_3',
#  'melt_proxy_3_4',
#  'melt_proxy_3_5',
#  'melt_proxy_3_6',
#  'melt_proxy_3_7',
#  'melt_proxy_4_1',
#  'melt_proxy_4_2',
#  'melt_proxy_4_3',
#  'melt_proxy_4_4',
#  'melt_proxy_4_5',
#  'melt_proxy_4_6',
#  'melt_proxy_4_7',
#  'melt_proxy_5_1',
#  'melt_proxy_5_2',
#  'melt_proxy_5_3',
#  'melt_proxy_5_4',
#  'melt_proxy_5_5',
#  'melt_proxy_5_6',
#  'melt_proxy_5_7',
#  'melt_proxy_6_1',
#  'melt_proxy_6_2',
#  'melt_proxy_6_3',
#  'melt_proxy_6_4',
#  'melt_proxy_6_5',
#  'melt_proxy_6_6',
#  'melt_proxy_6_7',
#  'melt_proxy_7_1',
#  'melt_proxy_7_2',
#  'melt_proxy_7_3',
#  'melt_proxy_7_4',
#  'melt_proxy_7_5',
#  'melt_proxy_7_6',
#  'melt_proxy_7_7'
# ),
#   trees = NULL
# )

In [ ]:
xgb1.feature_importances_

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Get scores and match them with feature names
feat_importances = pd.Series(xgb2.feature_importances_, index=X_train_ml.columns)

# 2. Plot the top 10 features
feat_importances.nlargest(20).sort_values().plot(kind='barh')
plt.title("Top 10 Important Features")
plt.show()
feat_importances

In [ ]:
import shap
# Xd = xgboost.DMatrix(X, label=y)

# explainer = shap.TreeExplainer(xgb1)
# explanation = explainer(X_train_ml)
# shap_values = explanation.values
explainer = shap.Explainer(xgb1)
shap_values = explainer(X_train_ml)

shap.plots.waterfall(shap_values[0])


In [ ]:
from shap import initjs


initjs()

In [ ]:
# visualize the first prediction's explanation with a force plot
shap.plots.force(shap_values[0])

In [ ]:
# visualize all the training set predictions
shap.plots.force(shap_values[:500])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the dark theme to match your MapLibre UI
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(12, 6), dpi=150)

# Use the test-series index (ensure datetime) as x-axis.
# df1 does not have a 'date' column because the dataframe was set_index('date') earlier.
date = pd.to_datetime(y_test_true.index)

# Ensure predictions are 1D arrays aligned with the date index
y_test_pred_plot = np.asarray(y_test_pred).ravel()
q_hybrid_plot = np.asarray(q_hybrid_testf).ravel()

# Plotting the three curves
ax.plot(date, y_test_true, label='Observed Discharge', color='#ffffff', linewidth=1.5, alpha=0.8)
ax.plot(date, y_test_pred_plot, label='Baseline Model', color='#ff4b4b', linestyle='--', linewidth=1.2)
ax.plot(date, q_hybrid_plot, label='Hybrid AI Model', color='#00d4ff', linewidth=2.5)

# Highlighting the "Peak Capture"
# You can shade the summer months where the SCA-melt logic is most active
# ax.axvspan('2023-06-01', '2023-08-31', color='#00d4ff', alpha=0.05, label='Peak Melt Season')

# Formatting
ax.set_title('Tarbela Inflow Prediction: Test Period Performance', loc='left', fontsize=16, pad=20)
ax.set_ylabel('Discharge ($m^3/s$)', fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(frameon=False, loc='upper right')

plt.tight_layout()
plt.savefig('hydrograph_performance.png', transparent=True)

In [ ]:

# In order of priority:

# Observed vs predicted hydrograph — most important, get this first
# Scatter plot with R² — quick to make once you have predictions exported
# Feature importance bar chart — you have the values already, just needs a clean plot
# Bias correction table — pull your Besham validation numbers